In [4]:
import numpy as np
import pandas as pd
import random
from xgboost import XGBRegressor

In [ ]:
class CausalContextualBandit:
    def __init__(self, treatments, epsilon=0.15):
        """
        treatments: Lista de assuntos possíveis (ex: ['pix', 'pagamento', 'seguro', 'investimento'])
        epsilon: Taxa de exploração (ex: 15% das vezes escolhe aleatoriamente para aprender)
        """
        self.treatments = treatments
        self.control_name = 'controle' # O braço silencioso
        self.epsilon = epsilon
        
        # Um modelo para cada tratamento. Ele vai prever o UPLIFT (CATE) contra o controle.
        self.models = {
            trt: XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.1) 
            for trt in treatments
        }
        
        # Flag para saber se os modelos já foram treinados pela primeira vez
        self.is_trained = {trt: False for trt in treatments}
        
    def recommend(self, context_features):
        """
        Recebe o contexto (features do cliente) e retorna a Ação (Assunto) e a Propensão.
        """
        arms = self.treatments + [self.control_name]
        n_arms = len(arms)
        
        # 1. Fase de Exploração (Random)
        if np.random.rand() < self.epsilon:
            chosen_arm = np.random.choice(arms)
            # A probabilidade de cair aqui é (epsilon * (1 / n_arms))
            propensity = self.epsilon / n_arms
            return chosen_arm, propensity
            
        # 2. Fase de Explotação (Usa os modelos)
        uplifts = {}
        for trt in self.treatments:
            if self.is_trained[trt]:
                # Prever o incremento (Uplift) que essa mensagem gera
                uplift_pred = self.models[trt].predict([context_features])[0]
            else:
                # Se ainda não tem dados, assume zero para forçar exploração
                uplift_pred = 0.0
            uplifts[trt] = uplift_pred
            
        # Encontra o tratamento com o maior uplift
        best_treatment = max(uplifts, key=uplifts.get)
        max_uplift = uplifts[best_treatment]
        
        # A DECISÃO CAUSAL: Se nenhuma mensagem gerar aumento de chance (uplift <= 0), 
        # a melhor ação é NÃO MANDAR NADA (Controle).
        if max_uplift <= 0:
            chosen_arm = self.control_name
        else:
            chosen_arm = best_treatment
            
        # A probabilidade de cair na explotação + a chance residual da exploração
        propensity = (1 - self.epsilon) + (self.epsilon / n_arms)
        
        return chosen_arm, propensity

    def train_batch(self, batch_data):
        """
        batch_data: DataFrame contendo o log do que aconteceu após os 7 dias.
        Colunas esperadas: features do contexto, 'action', 'propensity', 'converted'
        """
        # Para cada assunto, treinamos o modelo usando seus dados + os dados do grupo de controle
        for trt in self.treatments:
            # Filtra apenas quem recebeu ESTE tratamento ou o CONTROLE
            df_subset = batch_data[batch_data['action'].isin([trt, self.control_name])].copy()
            
            if len(df_subset) < 10: # Só treina se tiver o mínimo de dados
                continue
                
            # Cria a variável W (1 se foi Tratamento, 0 se foi Controle)
            df_subset['W'] = np.where(df_subset['action'] == trt, 1, 0)
            
            # -------------------------------------------------------------
            # O CORAÇÃO DO ALGORITMO: O RESULTADO TRANSFORMADO (Y*)
            # Isso transforma a conversão simples em "Uplift Incremental"
            # -------------------------------------------------------------
            y = df_subset['converted'].values
            w = df_subset['W'].values
            p = df_subset['propensity'].values
            
            # Proteção matemática para não dividir por zero
            p = np.clip(p, 0.01, 0.99) 
            
            # Fórmula do Y*: Y * ((W / p) - ((1 - W) / (1 - p)))
            y_star = y * ( (w / p) - ((1 - w) / (1 - p)) )
            
            # Features (Remove colunas de log)
            X = df_subset.drop(columns=['action', 'propensity', 'converted', 'W']).values
            
            # Treina o modelo XGBoost para prever o Y*
            self.models[trt].fit(X, y_star)
            self.is_trained[trt] = True
            
        print("Modelos de Uplift atualizados com sucesso!")

# ==========================================
# SIMULANDO O USO EM PRODUÇÃO
# ==========================================

# 1. Instanciamos nosso Agente Causal
treatments = ['pix', 'pagamento', 'seguro', 'investimento']
bandit = CausalContextualBandit(treatments=treatments, epsilon=0.20)

# 2. Quando o cliente abre a conta (Dia 1)
cliente_features = [25, 3500.0, 1] # Ex: Idade, Renda, Usa iOS (1=Sim)
assunto, propensao = bandit.recommend(cliente_features)

print(f"O motor escolheu: {assunto} (Probabilidade de escolha: {propensao:.2f})")
# AQUI VOCÊ PASSA 'assunto' PARA O SEU LLM GERAR A MENSAGEM (ou não gera nada se for controle).

# 3. Logamos isso no Banco de Dados (Ex: enviamos pro Kafka/Data Warehouse)
# log = {'idade': 25, 'renda': 3500, 'ios': 1, 'action': assunto, 'propensity': propensao, 'converted': ?}

# ... 7 DIAS SE PASSAM ...

# 4. Job Diário pega os clientes que bateram 7 dias e atualiza o modelo em Batch
# Simulação de um log do banco de dados (retorno dos 7 dias):
dados_historicos = pd.DataFrame({
    'idade': np.random.randint(18, 65, 1000),
    'renda': np.random.uniform(1000, 15000, 1000),
    'ios': np.random.randint(0, 2, 1000),
    'action': np.random.choice(['controle', 'pix', 'pagamento', 'seguro', 'investimento'], 1000),
    'propensity': [0.2] * 1000, # Supondo que foi totalmente exploratório no início
    'converted': np.random.randint(0, 2, 1000) # 1 se ativou, 0 se não ativou
})

print("\nRodando o job de atualização (Batch)...")
bandit.train_batch(dados_historicos)

# 5. Nova recomendação (Dia 8) com o modelo mais inteligente
novo_cliente = [45, 12000.0, 0] # Cliente mais velho, renda alta, Android
assunto, propensao = bandit.recommend(novo_cliente)
print(f"Para o novo cliente, o motor escolheu: {assunto}")